# Aula 09 · Parsers de texto

Esta aula apresenta o [capítulo 9 do site](https://lacouth.github.io/python_telecom-site/unidade4-texto/09-parsers-texto/). A ideia central: **a rede tem vários
fabricantes, e cada um escreve o mesmo evento do seu jeito**. A solução não é uma
ferramenta nova — é organizar o que você já sabe: **um leitor por formato, todos
entregando o mesmo dicionário**, e a linha que ninguém entende registrada, nunca
ignorada.

**Ao fim da aula você consegue:**

1. usar `partition`, `split(maxsplit=...)`, `startswith` e `isdigit` para
   desmontar uma linha sem quebrar a mensagem;
2. escrever um leitor por formato, com tabelas de tradução, que devolvem todos o
   mesmo dicionário;
3. detectar o formato de uma linha, validar um campo à mão e descartar com
   registro o que não se entende.

**Roteiro:** 🔥 aquecimento · 📟 chamado · 1. três fabricantes, um alarme · 2.
ferramentas novas · 3. um formato, uma função · 4. tabelas de tradução · 5.
descobrir o formato · 6. validar sem adivinhar · 7. a linha que não é de ninguém ·
📟 resolvendo o chamado · 🚪 antes de sair

## Como usar este caderno

- **Rode a célula ⚙️** logo abaixo antes de tudo (e de novo se o Colab reiniciar).
- **✍️ Passo:** o código é escrito ao vivo, em pedaços pequenos — a instrução
  está logo acima de cada célula vazia. Estudando sozinho, escreva você mesmo; o
  código completo está no capítulo do site (links 📖).
- Depois de escrever e **antes de rodar**, registre a sua previsão. Só então rode
  e abra o **▶ O que aconteceu**. A previsão errada é a parte que ensina — não a
  apague.
- **🎯 Sua vez:** escreva a solução e rode a célula `confere` logo abaixo dela —
  ✅ quer dizer que acertou, ❌ mostra o que ainda falta. A dica e uma solução
  estão recolhidas: tente antes de abrir.

In [ ]:
# ⚙️ Rode esta célula antes de tudo. Ela prepara a correção automática dos
# exercícios 🎯 — não precisa ler agora (usa coisas que só veremos mais tarde).
import math


def _mostra(argumentos):
    return ", ".join(repr(a) for a in argumentos)


def _igual(veio, esperado):
    if isinstance(esperado, float) and isinstance(veio, (int, float)):
        return math.isclose(veio, esperado, abs_tol=1e-9)
    return veio == esperado


def confere(funcao, casos):
    """Chama funcao com cada caso (argumentos, esperado) e diz se acertou."""
    certos = 0
    for numero, (argumentos, esperado) in enumerate(casos, start=1):
        chamada = f"{funcao.__name__}({_mostra(argumentos)})"
        try:
            veio = funcao(*argumentos)
        except Exception as erro:
            print(f"❌ {chamada} deu erro: {type(erro).__name__}: {erro}")
            continue
        if _igual(veio, esperado):
            certos += 1
            print(f"✅ {chamada} devolveu {veio!r}")
        elif veio is None:
            print(f"❌ {chamada} devolveu None — faltou o return?")
        else:
            print(f"❌ {chamada} devolveu {veio!r}, mas devia ser {esperado!r}")
    print(f"{certos} de {len(casos)} certos")


def confere_valor(nome, valor, esperado):
    """Diz se a variável `nome` ficou com o valor esperado."""
    if _igual(valor, esperado):
        print(f"✅ {nome} = {valor!r}")
    else:
        print(f"❌ {nome} vale {valor!r}, mas devia ser {esperado!r}")

## 🔥 Aquecimento — da aula passada

Sem rodar nada: quantos números são somados, e o que o programa imprime?

```python
textos = ["-21.4", "sem leitura", "-19.6"]
soma = 0
try:
    for t in textos:
        soma = soma + float(t)
except ValueError:
    pass
print(soma)
```

<details>
<summary><b>Resposta</b></summary>

Soma só o primeiro e imprime `-21.4`. O `try` está **em volta do laço**: no
`ValueError` de `"sem leitura"`, o Python sai do laço inteiro, e o `-19.6` nunca é
lido. Com o `try` **dentro** do laço, sairia `-41.0` — e hoje esse padrão volta,
para as linhas que nenhum leitor entende.

</details>

## 📟 O chamado de hoje

> **Chamado #0904 — NOC Maré Net**
>
> *"Estagiário, integramos os switches e os rádios novos à gerência. Problema: o
> painel só entende o formato de log das OLTs, e os alarmes dos outros
> equipamentos estão **invisíveis**. Preciso saber, para cada equipamento,
> **quantos alarmes graves** ele gerou — de qualquer fabricante — e **quantas
> linhas o sistema não conseguiu ler**."*

No fim da aula você consolida os três formatos num relatório só.

## 1. Três fabricantes, um alarme

O mesmo tipo de evento chega em três formatos: **A** (as OLTs, o formato do curso
até aqui), **B** (os switches, no estilo do syslog Cisco, com a severidade em
número de 0 a 7) e **C** (os rádios, no estilo chave=valor). Os três dizem as
mesmas quatro coisas — quando, quão grave, quem e o quê.

📖 [capítulo 9 · Três fabricantes, um alarme](https://lacouth.github.io/python_telecom-site/unidade4-texto/09-parsers-texto/#tres-fabricantes-um-alarme)

In [ ]:
# 📦 dados prontos — só rode esta célula
linha_a = "2026-03-02 14:03:17 CRITICAL OLT-CENTRO-01 perda de sinal na porta GPON0/1/3"
linha_b = "Mar  2 14:05:10 SWITCH-NORTE-02 %LINK-3-UPDOWN: Interface Gi0/12, changed state to down"
linha_c = 'ts=2026-03-02T14:07:44 host=RADIO-OESTE-01 sev=warn msg="enlace degradado"'
linhas = [linha_a, linha_b, linha_c]

**✍️ Passo 1.** Percorra `linhas` e imprima `linha.split()[2]` de cada uma — o jeito da Unidade 1
de pegar a severidade.

In [ ]:
# ✍️ passo 1

**Preveja:** as três linhas vão mostrar a severidade?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Só a primeira: `CRITICAL`. Na linha B a posição 2 é a hora (`14:05:10`); na C, é o
texto `sev=warn`. Cada fabricante põe as coisas num lugar diferente — o código de
posição fixa só funciona para um formato.

</details>

O objetivo da aula é transformar qualquer uma das três linhas **no mesmo
dicionário**, com as chaves `momento`, `severidade`, `equipamento` e `mensagem`.
Isso se chama **normalizar**.

## 2. Ferramentas novas para texto

`partition(sep)` corta na **primeira** ocorrência e devolve três pedaços: antes,
separador, depois. `split(maxsplit=N)` quebra no máximo N vezes. E `startswith`,
`isdigit` e `strip(caracteres)` perguntam sobre o texto sem quebrá-lo.

📖 [capítulo 9 · Ferramentas novas para texto](https://lacouth.github.io/python_telecom-site/unidade4-texto/09-parsers-texto/#ferramentas-novas-para-texto)

**✍️ Passo 2.** Primeiro o jeito que você já sabe: crie `par = "host=RADIO-OESTE-01"`, faça
`posicao = par.find("=")` e imprima `posicao`, `par[:posicao]` e `par[posicao + 1:]`.

In [ ]:
# ✍️ passo 2

**Preveja:** o que o `find` devolve?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`4` — a **posição** do `=` —, e as duas fatias dão `host` e `RADIO-OESTE-01`. Achar
o separador e fatiar dos dois lados funciona, mas são três linhas e uma conta de
`+ 1` para errar.

</details>

**✍️ Passo 3.** Agora o mesmo com `partition`: imprima `par.partition("=")` e depois desempacote
`chave, sep, valor = par.partition("=")`, imprimindo `chave` e `valor`.

In [ ]:
# ✍️ passo 3

**Preveja:** quantos pedaços o `partition` devolve?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Três: `('host', '=', 'RADIO-OESTE-01')`. O do meio é o próprio separador. Com o
desempacotamento, as três linhas do passo anterior viram uma.

</details>

**✍️ Passo 4.** Imprima `"-- MARK --".partition("=")` e `"-- MARK --".find("=")` — uma linha sem `=`.

In [ ]:
# ✍️ passo 4

**Preveja:** o `partition` dá erro quando o separador não existe?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Não dá erro: devolve `('-- MARK --', '', '')` — o texto inteiro no primeiro
pedaço, os outros dois vazios. O `find` devolve `-1`. Para dado sujo, "não quebrar
e deixar você conferir" é exatamente o que se quer.

</details>

**✍️ Passo 5.** Com a `linha_b` da célula de dados, imprima `linha_b.split(" ")[:4]` e, embaixo,
`linha_b.split()[:4]`. Repare nos **dois espaços** entre `Mar` e `2`.

In [ ]:
# ✍️ passo 5

**Preveja:** as duas linhas dão os mesmos quatro pedaços?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Não: `split(" ")` dá `['Mar', '', '2', '14:05:10']` — cada espaço separa, e dois
seguidos deixam um pedaço **vazio**. `split()` sem argumento trata qualquer
sequência de espaços como um separador só. **Para log, use `split()`.**

</details>

**✍️ Passo 6.** Faça `campos = linha_b.split(maxsplit=4)` e imprima `len(campos)` e `campos[4]`.

In [ ]:
# ✍️ passo 6

**Preveja:** o que fica em `campos[4]`?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`5` pedaços, e `campos[4]` é **o resto inteiro da linha**:
`%LINK-3-UPDOWN: Interface Gi0/12, changed state to down`, com os espaços. O
`maxsplit` para de quebrar depois de 4 cortes — a mensagem não se despedaça.

</details>

**✍️ Passo 7.** Imprima `linha_c.startswith("ts=")`, `"2026".isdigit()`, `"-5".isdigit()` e
`'"enlace"'.strip('"')`.

In [ ]:
# ✍️ passo 7

**Preveja:** `"-5".isdigit()` dá `True` ou `False`?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`True`, `True`, **`False`** e `enlace`. O sinal de menos não é algarismo — guarde
isso para a validação. E `strip` com argumento tira **esses** caracteres das
pontas, em vez de espaços.

</details>

### 🎯 Sua vez — A interface da mensagem

Mensagens de switch trazem a interface assim: `"Interface Gi0/12, changed state to down"`.
Escreva `interface_da_mensagem(mensagem)`, que devolve o nome da interface
(`"Gi0/12"`), ou `None` se a mensagem não começar com `"Interface "`.

In [ ]:
def interface_da_mensagem(mensagem):
    # sua solução aqui
    pass

In [ ]:
confere(interface_da_mensagem, [
    (("Interface Gi0/12, changed state to down",), "Gi0/12"),
    (("Interface Te1/0/1, changed state to up",), "Te1/0/1"),
    (("Configured from console by admin",), None),
])

<details>
<summary><b>💡 Dica</b></summary>

Confira o começo com `startswith`. Depois, dois `partition`: um no espaço (para
jogar fora a palavra `Interface`) e outro na vírgula (para ficar com o que vem
antes dela).

</details>

## 3. Um formato, uma função

Cada formato ganha **uma função**, e as três devolvem o mesmo dicionário. O A é o
mais simples — e o `maxsplit` resolve a mensagem com espaços, que no capítulo 2
exigia um `join`.

📖 [capítulo 9 · Um formato, uma função](https://lacouth.github.io/python_telecom-site/unidade4-texto/09-parsers-texto/#um-formato-uma-funcao)

**✍️ Passo 8.** Escreva `le_formato_a(linha)`: `campos = linha.split(maxsplit=4)` e devolva um
dicionário com `"momento"` (`campos[0] + " " + campos[1]`), `"severidade"`
(`campos[2]`), `"equipamento"` (`campos[3]`) e `"mensagem"` (`campos[4]`). Imprima
`le_formato_a(linha_a)`.

In [ ]:
# ✍️ passo 8

**Preveja:** o que fica na chave `"mensagem"`?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

A mensagem inteira, `perda de sinal na porta GPON0/1/3`, com os espaços. É o
primeiro leitor: entra uma linha do formato A, sai o dicionário normalizado.

</details>

## 4. Tabelas de tradução

Cada fabricante diz a mesma coisa com outras palavras: o B escreve a severidade
como número e o mês como `Mar`; o C escreve `warn`. Traduzir é trabalho para um
**dicionário** usado como tabela. Rode a célula abaixo — as tabelas prontas.

📖 [capítulo 9 · Tabelas de tradução](https://lacouth.github.io/python_telecom-site/unidade4-texto/09-parsers-texto/#tabelas-de-traducao)

In [ ]:
# 📦 dados prontos — só rode esta célula
MESES = {"Jan": 1, "Feb": 2, "Mar": 3, "Apr": 4, "May": 5, "Jun": 6,
         "Jul": 7, "Aug": 8, "Sep": 9, "Oct": 10, "Nov": 11, "Dec": 12}
SEVERIDADE_POR_NUMERO = {"0": "EMERGENCY", "1": "ALERT", "2": "CRITICAL", "3": "ERROR",
                         "4": "WARNING", "5": "NOTICE", "6": "INFO", "7": "DEBUG"}
SEVERIDADE_C = {"crit": "CRITICAL", "warn": "WARNING", "info": "INFO"}

**✍️ Passo 9.** Traduza o código: `partes = "%LINK-3-UPDOWN".strip("%").split("-")`, imprima
`partes` e depois `SEVERIDADE_POR_NUMERO[partes[1]]`.

In [ ]:
# ✍️ passo 9

**Preveja:** que severidade é o número 3?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`['LINK', '3', 'UPDOWN']` e `ERROR`. As chaves da tabela são **texto** (`"3"`): o
que sai do `split` é texto, e a consulta fica direta, sem `int()`. Os nomes em
maiúsculas são a convenção para "tabela fixa".

</details>

**✍️ Passo 10.** Escreva `le_formato_b(linha, ano=2026)`: `campos = linha.split(maxsplit=4)`;
`mes = MESES[campos[0]]`; `dia = int(campos[1])`;
`codigo, _, mensagem = campos[4].partition(": ")`; e o número da severidade no
meio do código, como no passo anterior. O momento é
`f"{ano}-{mes:02d}-{dia:02d} {campos[2]}"`. Imprima `le_formato_b(linha_b)`.

In [ ]:
# ✍️ passo 10

**Preveja:** de onde sai o ano, se a linha não tem ano?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Do **parâmetro** com valor padrão (Aula 04): o formato B não grava o ano, e quem
chama decide. Sai `2026-03-02 14:05:10`, `ERROR`, `SWITCH-NORTE-02` e a mensagem
depois do primeiro `": "`. O `:02d` transforma o 3 em `03`.

</details>

### 🎯 Sua vez — O momento do formato B

Escreva `momento_b(mes, dia, hora, ano)`, que recebe os pedaços como vêm do
formato B (mês abreviado e dia em texto) e devolve o momento normalizado.

In [ ]:
def momento_b(mes, dia, hora, ano):
    # sua solução aqui
    pass

In [ ]:
confere(momento_b, [
    (("Mar", "2", "14:05:10", 2026), "2026-03-02 14:05:10"),
    (("Dec", "31", "23:59:58", 2025), "2025-12-31 23:59:58"),
    (("Jan", "9", "00:00:01", 2026), "2026-01-09 00:00:01"),
])

<details>
<summary><b>💡 Dica</b></summary>

O mês passa pela tabela `MESES` (rode a célula de dados). O dia vem como texto:
`int(dia)`. E o `:02d` completa com zero à esquerda.

</details>

**✍️ Passo 11.** Escreva `le_formato_c(linha)`. **Primeiro** separe a mensagem:
`antes, _, mensagem = linha.partition(" msg=")`. Depois monte `campos = {}` e,
para cada `par` em `antes.split()`, faça `chave, _, valor = par.partition("=")` e
`campos[chave] = valor`. Devolva o dicionário normalizado: momento com
`campos["ts"].replace("T", " ")`, severidade pela `SEVERIDADE_C`, e a mensagem com
`strip('"')`. Imprima `le_formato_c(linha_c)`.

In [ ]:
# ✍️ passo 11

**Preveja:** por que separar a mensagem **antes** do `split()`?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Porque a mensagem tem espaços **dentro das aspas**: um `split()` na linha inteira
cortaria `"enlace degradado"` em dois pedaços, e o segundo (`degradado"`) não teria
`=`. Com o `partition` primeiro, o resto só tem pares `chave=valor`, e sai
`{'momento': '2026-03-02 14:07:44', 'severidade': 'WARNING', ...}`.

</details>

## 5. Descobrir o formato e despachar

Falta decidir qual leitor chamar. Cada formato tem um sinal logo no começo: C começa
com `ts=`; B começa com um mês e tem `%`; A começa com quatro algarismos e um
hífen.

📖 [capítulo 9 · Descobrir o formato e despachar](https://lacouth.github.io/python_telecom-site/unidade4-texto/09-parsers-texto/#descobrir-o-formato-e-despachar)

**✍️ Passo 12.** Escreva `detecta_formato(linha)`: **se** `linha.startswith("ts=")`, devolva `"C"`;
**se** `linha[:3] in MESES and "%" in linha`, `"B"`; **se**
`linha[:4].isdigit() and linha[4:5] == "-"`, `"A"`; senão, `None`. Percorra
`linhas + ["-- MARK --", ""]` imprimindo o formato de cada uma.

In [ ]:
# ✍️ passo 12

**Preveja:** o que sai para `-- MARK --` e para a linha vazia?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`A`, `B`, `C`, `None` e `None`. O `-- MARK --` é uma linha que servidores de syslog
escrevem só para dizer "estou vivo" — não é de fabricante nenhum. Repare no
`linha[4:5]` em vez de `linha[4]`: fatia de texto vazio dá `""`, índice daria
`IndexError`.

</details>

> ⚠️ **Armadilha.** Testar o sinal mais genérico primeiro. "Começa com algarismo" também seria verdade
> para outras coisas; por isso os testes vão do mais específico (`ts=`) para o mais
> genérico, e cada um confere **dois** sinais quando um só é ambíguo (`Mar` pode
> ser o começo de `Marcos`, mas não com `%` na linha).

### 🎯 Sua vez — Um quarto fabricante

Um fornecedor novo manda linhas em JSON, que começam com `{`. Escreva
`detecta_formato_v2(linha)`, que devolve `"A"`, `"B"`, `"C"`, `"D"` (para as que
começam com `{`) ou `None`.

In [ ]:
def detecta_formato_v2(linha):
    # sua solução aqui
    pass

In [ ]:
confere(detecta_formato_v2, [
    ((linha_a,), "A"),
    ((linha_b,), "B"),
    ((linha_c,), "C"),
    (('{"host": "OLT-9", "sev": "crit"}',), "D"),
    (("Marcos esteve aqui",), None),
    (("",), None),
])

<details>
<summary><b>💡 Dica</b></summary>

É o `detecta_formato` do passo, com mais um `if`. A tabela `MESES` está na célula
de dados da seção 4.

</details>

## 6. Validar sem adivinhar

Parte do trabalho com texto é **recusar** o que não está no formato, antes que o
dado errado entre no relatório. Um IPv4 são quatro números de 0 a 255 separados por
ponto; cada condição vira uma linha, e a função devolve `False` na primeira que
falhar.

📖 [capítulo 9 · Validar sem adivinhar](https://lacouth.github.io/python_telecom-site/unidade4-texto/09-parsers-texto/#validar-sem-adivinhar)

**✍️ Passo 13.** Escreva `ip_valido(texto)`: `partes = texto.strip().split(".")`; **se**
`len(partes) != 4`, `return False`; para cada `parte`, **se não**
`parte.isdigit()`, `return False`, e **se** `int(parte) > 255`, `return False`; no
fim, `return True`. Teste com `"10.0.3.47"`, `"10.0.3.256"`, `"10.0..47"` e
`"10.0.3.-1"`.

In [ ]:
# ✍️ passo 13

**Preveja:** qual condição derruba cada um dos três últimos?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`True`, depois três `False`, cada um por uma condição: `256` passa do limite;
`10.0..47` tem uma parte vazia (e `"".isdigit()` é `False`); `-1` tem um sinal que
não é algarismo. **Os casos de borda são o teste de verdade de um validador** —
testado só com `"10.0.3.47"`, ele não foi testado.

</details>

> ⚠️ **Armadilha.** Escrever `int(parte)` antes de conferir `isdigit()`. Com `"dez"`, o `int` dá
> `ValueError` e derruba o programa em vez de devolver `False`. A ordem das
> condições é parte da lógica: primeiro "é número?", depois "que número?".

### 🎯 Sua vez — Porta válida

Escreva `porta_valida(texto)`, que devolve `True` se o texto é uma porta TCP/UDP
válida — um número inteiro de **1 a 65535** — e `False` caso contrário.

In [ ]:
def porta_valida(texto):
    # sua solução aqui
    pass

In [ ]:
confere(porta_valida, [
    (("22",), True),
    (("65535",), True),
    (("0",), False),
    (("65536",), False),
    (("-1",), False),
    (("ssh",), False),
    ((" 443 ",), True),
])

<details>
<summary><b>💡 Dica</b></summary>

Mesma ordem do `ip_valido`: `strip`, depois `isdigit()`, só então `int()` e a
faixa.

</details>

O capítulo também valida **MAC** (seis pares hexadecimais, com `:` ou `-`),
normalizando antes de conferir.

## 7. A linha que não é de ninguém

Um log real traz linhas que nenhum leitor entende: a de formato desconhecido, a
cortada no meio, a com um campo corrompido. O padrão é o da Aula 08: descartar,
**registrar** e seguir.

📖 [capítulo 9 · A linha que não é de ninguém](https://lacouth.github.io/python_telecom-site/unidade4-texto/09-parsers-texto/#a-linha-que-nao-e-de-ninguem)

**✍️ Passo 14.** Escreva `le_linha(linha)`: `formato = detecta_formato(linha)`; dentro de um `try:`,
chame `le_formato_a`, `le_formato_b` ou `le_formato_c` conforme o formato (um `if`
com `return` para cada); `except (IndexError, KeyError, ValueError): return None`;
e, depois do `try`, `return None`. Teste com
`le_linha("Mar  2 14:20:31 SW-1 %LINK-X-UPDOWN: Gi0/1 down")`.

In [ ]:
# ✍️ passo 14

**Preveja:** essa linha tem formato B. O que acontece com o `X` no lugar do número?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Devolve `None`. A linha **parece** B — `detecta_formato` diz `"B"` —, mas dentro do
leitor a tabela não tem a chave `"X"`, e o `KeyError` é capturado. Os erros entre
parênteses no `except` são os três tipos que uma linha quebrada provoca: campo
faltando (`IndexError`), chave que não existe (`KeyError`), número que não é
número (`ValueError`).

</details>

> ⚠️ **Armadilha.** Descartar sem contar. Se um fabricante muda o formato numa atualização de firmware,
> **todas** as linhas dele caem no descarte — e, sem a contagem no relatório, ele
> simplesmente some, sem nenhum erro na tela. Por isso o resumo sempre diz quantas
> linhas não foram lidas.

## 📟 Resolvendo o chamado

Rode as duas células abaixo: o log do dia, com os três formatos misturados, e os
leitores prontos (os mesmos que você escreveu, para o exercício rodar sozinho).

In [ ]:
# 📦 dados prontos — só rode esta célula
log_do_dia = [
    "2026-03-02 14:03:17 CRITICAL OLT-CENTRO-01 perda de sinal na porta GPON0/1/3",
    "Mar  2 14:05:10 SWITCH-NORTE-02 %LINK-3-UPDOWN: Interface Gi0/12, changed state to down",
    'ts=2026-03-02T14:07:44 host=RADIO-OESTE-01 sev=warn msg="enlace degradado"',
    "-- MARK --",
    "Mar  2 14:20:31 SWITCH-NORTE-02 %LINK-X-UPDOWN: Interface Gi0/12, changed state to up",
    "2026-03-02 14:31:02 WARNING OLT-SUL-03 potencia optica degradada: -27.2 dBm",
    "ts=2026-03-02T14:40:00 host=RADIO-OESTE-01",
    "Mar  2 14:52:16 SWITCH-CENTRO-01 %SYS-2-MALLOCFAIL: Memory allocation of 4096 bytes failed",
    "2026-03-02 15:0",
    'ts=2026-03-02T15:10:09 host=RADIO-OESTE-01 sev=crit msg="enlace fora do ar"',
    "Mar  2 15:22:40 SWITCH-NORTE-02 %LINK-3-UPDOWN: Interface Gi0/7, changed state to down",
    "2026-03-02 15:30:12 CRITICAL OLT-CENTRO-01 equipamento sem resposta ha 4 minutos",
]

In [ ]:
# Os leitores da aula, prontos — só rode esta célula.
def le_formato_a(linha):
    campos = linha.split(maxsplit=4)
    return {"momento": campos[0] + " " + campos[1], "severidade": campos[2],
            "equipamento": campos[3], "mensagem": campos[4]}


def le_formato_b(linha, ano=2026):
    campos = linha.split(maxsplit=4)
    mes = MESES[campos[0]]
    dia = int(campos[1])
    codigo, _, mensagem = campos[4].partition(": ")
    partes = codigo.strip("%").split("-")
    return {"momento": f"{ano}-{mes:02d}-{dia:02d} {campos[2]}",
            "severidade": SEVERIDADE_POR_NUMERO[partes[1]],
            "equipamento": campos[3], "mensagem": mensagem}


def le_formato_c(linha):
    antes, _, mensagem = linha.partition(" msg=")
    campos = {}
    for par in antes.split():
        chave, _, valor = par.partition("=")
        campos[chave] = valor
    return {"momento": campos["ts"].replace("T", " "),
            "severidade": SEVERIDADE_C[campos["sev"]],
            "equipamento": campos["host"], "mensagem": mensagem.strip('"')}


def detecta_formato(linha):
    if linha.startswith("ts="):
        return "C"
    if linha[:3] in MESES and "%" in linha:
        return "B"
    if linha[:4].isdigit() and linha[4:5] == "-":
        return "A"
    return None


def le_linha(linha):
    """Alarme normalizado, ou None se a linha não for de ninguém."""
    formato = detecta_formato(linha)
    try:
        if formato == "A":
            return le_formato_a(linha)
        if formato == "B":
            return le_formato_b(linha)
        if formato == "C":
            return le_formato_c(linha)
    except (IndexError, KeyError, ValueError):
        return None
    return None

### 🎯 Sua vez — Os alarmes graves de cada equipamento

Escreva `graves_por_equipamento(linhas)`, que devolve **dois valores**:

1. um dicionário **equipamento → quantidade de alarmes graves**, em que grave é
   severidade `EMERGENCY`, `ALERT`, `CRITICAL` ou `ERROR`;
2. **quantas** linhas não puderam ser lidas (`le_linha` devolveu `None`).

Use o `le_linha` da célula acima.

In [ ]:
GRAVES = {"EMERGENCY", "ALERT", "CRITICAL", "ERROR"}


def graves_por_equipamento(linhas):
    # sua solução aqui
    pass

In [ ]:
confere(graves_por_equipamento, [
    ((log_do_dia,), ({"OLT-CENTRO-01": 2, "SWITCH-NORTE-02": 2,
                      "SWITCH-CENTRO-01": 1, "RADIO-OESTE-01": 1}, 4)),
    ((["-- MARK --", "2026-03-02 09:00:00 INFO X ok"],), ({}, 1)),
])

<details>
<summary><b>💡 Dica</b></summary>

Um laço só: `alarme = le_linha(linha)`; se for `None`, some 1 aos não lidos e
`continue`; se `alarme["severidade"] in GRAVES`, conte o equipamento com o
`.get(nome, 0) + 1` da Aula 05. No fim, `return contagem, nao_lidas`.

</details>

**Resposta ao chamado:** os switches e o rádio, que estavam invisíveis no painel,
somam 4 dos 6 alarmes graves do dia. E 4 linhas não puderam ser lidas — número
que precisa aparecer no relatório, porque é ele que avisa quando um fabricante muda
o formato.

## 🚪 Antes de sair

Responda de cabeça, sem rodar.

**1.** `"a=b=c".partition("=")` devolve:
a) `('a', 'b', 'c')`  b) `('a', '=', 'b=c')`  c) `['a', 'b=c']`  d) erro

<details>
<summary><b>Resposta da 1</b></summary>

**b**. O `partition` corta só na **primeira** ocorrência, e devolve sempre três
pedaços, com o separador no meio.

</details>

**2.** Em `"Mar  2 14:05:10"` (dois espaços depois de `Mar`), `split(" ")[1]` vale:
a) `"2"`  b) `" 2"`  c) `""`  d) `"14:05:10"`

<details>
<summary><b>Resposta da 2</b></summary>

**c**. Cada espaço separa, e dois seguidos deixam um pedaço vazio. Com `split()`,
a posição 1 seria `"2"`.

</details>

**3.** Um `ip_valido` testado só com `"10.0.3.47"` e `"abc"` pode estar errado em:
a) nada, os dois casos cobrem tudo  b) `"256.1.1.1"`  c) `"10.0.3.47"`  d) `"abc"`

<details>
<summary><b>Resposta da 3</b></summary>

**b**. Sem um caso de borda para a faixa, um validador que esqueceu o `<= 255`
passa nos dois testes.

</details>

## 🏠 Para casa

- [Lista 09](https://lacouth.github.io/python_telecom-site/listas/lista09/) —
  parsers de texto, com testes automáticos no Colab.
- Releia o [capítulo 9 do site](https://lacouth.github.io/python_telecom-site/unidade4-texto/09-parsers-texto/), principalmente a validação de MAC.
- **Na próxima aula:** mini-teste sobre esta aula (`partition`, `split` e
  validação).